In [9]:
import requests
import json
from datetime import datetime

# API Keys
POLYGON_API_KEY = "mxHpmdO4wzkVJhzExKhIfbXbUDr0OmCW"
EODHD_API_KEY = "67ffece4b2ae08.94077168"

def get_eodhd_fundamentals(ticker):
    """Fetch fundamental data from EODHD API"""
    url = f"https://eodhd.com/api/fundamentals/{ticker}.US"
    params = {
        "api_token": EODHD_API_KEY,
        "fmt": "json"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        print("\n=== EODHD API Response ===")
        
        # Check for cash flow data
        if "Financials" in data and "Cash_Flow" in data["Financials"]:
            cash_flow = data["Financials"]["Cash_Flow"]
            print(f"\nAnnual Cash Flow Data for {ticker}:")
            print("-" * 80)
            
            # Get yearly data
            yearly_data = cash_flow.get("yearly", {})
            
            # Sort years in descending order
            sorted_years = sorted(yearly_data.keys(), reverse=True)
            
            for year in sorted_years:
                year_data = yearly_data[year]
                operating_cf = year_data.get("operatingCashFlow", "N/A")
                
                # Try to get shares outstanding
                shares = None
                if "SharesStats" in data and "SharesOutstanding" in data["SharesStats"]:
                    shares = data["SharesStats"]["SharesOutstanding"]
                
                print(f"\nYear: {year}")
                print(f"  Operating Cash Flow: ${operating_cf:,}" if isinstance(operating_cf, (int, float)) else f"  Operating Cash Flow: {operating_cf}")
                
                if shares and isinstance(operating_cf, (int, float)):
                    cf_per_share = operating_cf / shares
                    print(f"  Shares Outstanding: {shares:,}")
                    print(f"  Cash Flow Per Share: ${cf_per_share:.2f}")
                
            return data
        else:
            print("Cash flow data not found in response")
            return data
            
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from EODHD: {e}")
        return None

def get_polygon_financials(ticker):
    """Fetch financial data from Polygon API"""
    url = f"https://api.polygon.io/vX/reference/financials"
    params = {
        "ticker": ticker,
        "timeframe": "annual",
        "limit": 10,
        "apiKey": POLYGON_API_KEY
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        print("\n=== Polygon API Response ===")
        
        if "results" in data and len(data["results"]) > 0:
            print(f"\nAnnual Cash Flow Per Share for {ticker}:")
            print("-" * 80)
            
            for result in data["results"]:
                fiscal_period = result.get("fiscal_period", "N/A")
                fiscal_year = result.get("fiscal_year", "N/A")
                start_date = result.get("start_date", "N/A")
                end_date = result.get("end_date", "N/A")
                
                # Access cash flow statement
                cash_flow = result.get("financials", {}).get("cash_flow_statement", {})
                balance_sheet = result.get("financials", {}).get("balance_sheet", {})
                
                operating_cf = cash_flow.get("net_cash_flow_from_operating_activities", {}).get("value")
                shares = balance_sheet.get("equity_attributable_to_parent", {}).get("value")
                
                # Try to get basic shares outstanding
                if not shares:
                    shares = balance_sheet.get("common_stock_shares_outstanding", {}).get("value")
                
                print(f"\nFiscal Year: {fiscal_year} (Period: {fiscal_period})")
                print(f"  Period: {start_date} to {end_date}")
                
                if operating_cf:
                    print(f"  Operating Cash Flow: ${operating_cf:,.0f}")
                    
                if shares and operating_cf:
                    cf_per_share = operating_cf / shares
                    print(f"  Shares Outstanding: {shares:,.0f}")
                    print(f"  Cash Flow Per Share: ${cf_per_share:.2f}")
                elif operating_cf:
                    print(f"  Shares Outstanding: Data not available")
                    print(f"  Note: Cannot calculate per-share value without shares outstanding")
                    
            return data
        else:
            print("No financial data found in Polygon response")
            return data
            
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from Polygon: {e}")
        return None

if __name__ == "__main__":
    ticker = "AMD"
    
    print(f"Fetching annual cash flow per share data for {ticker}...")
    print("=" * 80)
    
    # Try EODHD first
    eodhd_data = get_eodhd_fundamentals(ticker)
    
    # Try Polygon
    polygon_data = get_polygon_financials(ticker)
    
    print("\n" + "=" * 80)
    print("Data fetch complete!")

ImportError: cannot import name '__version__' from 'urllib3' (unknown location)

In [2]:
!pip install requests